# Enterprise Credit Pricing Pipelines: SAS to Databricks Migration

This advanced notebook uses financial credit-pricing data to show production-style Databricks ML patterns for SAS migration.

You will learn how to:
- Generate distributed credit application data
- Build cross-validated pricing models
- Evaluate model performance on test data
- Add polynomial features for nonlinear risk pricing
- Capture model metadata for production review


## 1. Import Libraries

The imports are explicit so learners can see where each class comes from.


In [ ]:
import json
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import PolynomialExpansion, StandardScaler, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

print("Enterprise credit pricing notebook")
print(f"Spark version: {spark.version}")
print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## 2. Create Distributed Credit Pricing Data

Spark creates the data across the cluster. The target is `interest_rate_percent`, calculated from credit score, income, loan amount, debt-to-income, utilization, delinquencies, term, and prior default.


In [ ]:
row_count = 50_000

credit_applications = (
    spark.range(row_count)
    .withColumn("credit_score", (F.rand(seed=42) * 331 + 520).cast("int"))
    .withColumn("annual_income", (F.rand(seed=43) * 195000 + 25000).cast("double"))
    .withColumn("loan_amount", (F.rand(seed=44) * 70000 + 5000).cast("double"))
    .withColumn("debt_to_income", (F.rand(seed=45) * 0.50 + 0.05).cast("double"))
    .withColumn("credit_utilization", (F.rand(seed=46) * 0.90 + 0.05).cast("double"))
    .withColumn("delinquencies", (F.rand(seed=47) * 5).cast("int"))
    .withColumn("loan_term_months", (F.floor(F.rand(seed=48) * 4) * 12 + 36).cast("int"))
    .withColumn("prior_default", (F.rand(seed=49) > 0.88).cast("int"))
)

credit_applications = credit_applications.withColumn(
    "score_risk",
    F.greatest((F.lit(720) - F.col("credit_score")) / F.lit(100), F.lit(0)),
)

credit_applications = credit_applications.withColumn(
    "interest_rate_percent",
    4.5
    + 1.8 * F.col("score_risk")
    + 2.2 * F.col("score_risk") * F.col("score_risk")
    + 6.0 * F.col("debt_to_income")
    + 3.0 * F.col("credit_utilization")
    + 0.45 * F.col("delinquencies")
    + 1.2 * F.col("prior_default")
    + 0.02 * (F.col("loan_term_months") - 36)
    + 2.0 * (F.col("loan_amount") / F.col("annual_income"))
    + F.rand(seed=50) * 1.2,
).drop("id", "score_risk")

credit_applications = credit_applications.cache()

print(f"Rows: {credit_applications.count():,}")
print(f"Partitions: {credit_applications.rdd.getNumPartitions()}")
credit_applications.show(5)


## 3. SAS Reference: Large Credit Data Generation

SAS can create similar rows in a `DATA` step, but Spark is built to distribute this work across a cluster.

```sas
DATA credit_applications;
  DO application_id = 1 TO 50000;
    credit_score = 520 + FLOOR(RAND('UNIFORM') * 331);
    annual_income = 25000 + RAND('UNIFORM') * 195000;
    loan_amount = 5000 + RAND('UNIFORM') * 70000;
    debt_to_income = 0.05 + RAND('UNIFORM') * 0.50;
    credit_utilization = 0.05 + RAND('UNIFORM') * 0.90;
    prior_default = (RAND('UNIFORM') > 0.88);
    interest_rate_percent = 4.5 + 6*debt_to_income + 3*credit_utilization;
    OUTPUT;
  END;
RUN;
```


## 4. Visualize a Sample from Distributed Data

Large Spark DataFrames should not be fully converted to pandas. Take a small sample before plotting.


In [ ]:
sample_data = credit_applications.select(
    "credit_score",
    "debt_to_income",
    "interest_rate_percent",
).sample(fraction=0.02, seed=42).limit(1000).toPandas()

plt.figure(figsize=(8, 5))
scatter = plt.scatter(
    sample_data["credit_score"],
    sample_data["interest_rate_percent"],
    c=sample_data["debt_to_income"],
    alpha=0.4,
)
plt.title("Sample Credit Pricing Data")
plt.xlabel("Credit Score")
plt.ylabel("Interest Rate (%)")
plt.colorbar(scatter, label="Debt-to-Income")
plt.grid(True)
plt.show()


## 5. Build a Cross-Validated Pricing Pipeline

Cross-validation trains several versions of the model and chooses the best settings using a metric. Here we optimize for R2.


In [ ]:
train_data, test_data = credit_applications.randomSplit([0.7, 0.3], seed=42)

feature_columns = [
    "credit_score",
    "annual_income",
    "loan_amount",
    "debt_to_income",
    "credit_utilization",
    "delinquencies",
    "loan_term_months",
    "prior_default",
]
target_column = "interest_rate_percent"

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=False, withStd=True)
regression = LinearRegression(featuresCol="scaled_features", labelCol=target_column)

pipeline = Pipeline(stages=[assembler, scaler, regression])

parameter_grid = (
    ParamGridBuilder()
    .addGrid(regression.regParam, [0.0, 0.01, 0.1])
    .addGrid(regression.elasticNetParam, [0.0, 0.5])
    .build()
)

r2_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="r2")

cross_validator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=parameter_grid,
    evaluator=r2_evaluator,
    numFolds=3,
    parallelism=2,
)

print(f"Parameter combinations: {len(parameter_grid)}")
print("Training cross-validated pricing model...")

started_at = datetime.now()
cross_validated_model = cross_validator.fit(train_data)
elapsed_seconds = (datetime.now() - started_at).total_seconds()

best_regression = cross_validated_model.bestModel.stages[-1]

print(f"Training finished in {elapsed_seconds:.1f} seconds")
print(f"Best regParam: {best_regression.getRegParam()}")
print(f"Best elasticNetParam: {best_regression.getElasticNetParam()}")


## 6. SAS Reference: Manual Pricing Parameter Search

SAS users often compare model choices with multiple procedure runs. PySpark `CrossValidator` automates that pattern.

```sas
PROC GLMSELECT DATA=train_data;
  MODEL interest_rate_percent = credit_score annual_income loan_amount debt_to_income
      credit_utilization delinquencies loan_term_months prior_default
    / SELECTION=LASSO(CHOOSE=CV);
RUN;
```


## 7. Evaluate the Best Pricing Model

These metrics can be logged with every model training run.


In [ ]:
test_predictions = cross_validated_model.transform(test_data)

rmse_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="rmse")
mae_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="mae")

best_test_rmse = rmse_evaluator.evaluate(test_predictions)
best_test_mae = mae_evaluator.evaluate(test_predictions)
best_test_r2 = r2_evaluator.evaluate(test_predictions)

print("Best model test metrics")
print(f"RMSE: {best_test_rmse:,.4f}")
print(f"MAE:  {best_test_mae:,.4f}")
print(f"R2:   {best_test_r2:.4f}")


## 8. Visualize Best Model Predictions

Dots close to the diagonal line are better predictions.


In [ ]:
actual_vs_predicted = test_predictions.select(target_column, "prediction").limit(1000).toPandas()

min_rate = min(actual_vs_predicted[target_column].min(), actual_vs_predicted["prediction"].min())
max_rate = max(actual_vs_predicted[target_column].max(), actual_vs_predicted["prediction"].max())

plt.figure(figsize=(6, 6))
plt.scatter(actual_vs_predicted[target_column], actual_vs_predicted["prediction"], alpha=0.35)
plt.plot([min_rate, max_rate], [min_rate, max_rate], color="red", label="Perfect prediction")
plt.title("Cross-Validated Pricing Model: Actual vs Predicted")
plt.xlabel("Actual Interest Rate (%)")
plt.ylabel("Predicted Interest Rate (%)")
plt.legend()
plt.grid(True)
plt.show()


## 9. Add Polynomial Features for Nonlinear Risk Pricing

Polynomial features help capture nonlinear risk pricing and interactions between credit variables.


In [ ]:
polynomial_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
polynomial_expansion = PolynomialExpansion(degree=2, inputCol="features", outputCol="polynomial_features")
polynomial_scaler = StandardScaler(inputCol="polynomial_features", outputCol="scaled_polynomial_features", withMean=False, withStd=True)
polynomial_regression = LinearRegression(
    featuresCol="scaled_polynomial_features",
    labelCol=target_column,
    regParam=0.01,
)

polynomial_pipeline = Pipeline(stages=[
    polynomial_assembler,
    polynomial_expansion,
    polynomial_scaler,
    polynomial_regression,
])

print("Training polynomial pricing model...")
polynomial_model = polynomial_pipeline.fit(train_data)

polynomial_predictions = polynomial_model.transform(test_data)
polynomial_rmse = rmse_evaluator.evaluate(polynomial_predictions)
polynomial_r2 = r2_evaluator.evaluate(polynomial_predictions)

original_feature_count = len(feature_columns)
polynomial_feature_count = int((original_feature_count + 1) * (original_feature_count + 2) / 2 - 1)

print(f"Original feature count: {original_feature_count}")
print(f"Polynomial feature count: {polynomial_feature_count}")
print(f"Polynomial RMSE: {polynomial_rmse:,.4f}")
print(f"Polynomial R2: {polynomial_r2:.4f}")


## 10. Visualize the Credit Score Pricing Curve

This chart holds other borrower values fixed and changes only credit score. It shows how polynomial features can learn a curve.


In [ ]:
credit_score_grid = spark.createDataFrame([
    (score, 85000.0, 30000.0, 0.28, 0.35, 0, 60, 0)
    for score in range(520, 851, 5)
], feature_columns)

linear_curve = cross_validated_model.transform(credit_score_grid).select("credit_score", "prediction").toPandas()
polynomial_curve = polynomial_model.transform(credit_score_grid).select("credit_score", "prediction").toPandas()

plt.figure(figsize=(9, 5))
plt.plot(linear_curve["credit_score"], linear_curve["prediction"], label="Cross-validated linear model")
plt.plot(polynomial_curve["credit_score"], polynomial_curve["prediction"], label="Polynomial model")
plt.title("Enterprise Credit Score Pricing Curve")
plt.xlabel("Credit Score")
plt.ylabel("Predicted Interest Rate (%)")
plt.legend()
plt.grid(True)
plt.show()


## 11. Visualize Production Model Metrics

Use metric charts in model reviews to decide whether extra complexity is worth it.


In [ ]:
model_names = ["Cross-Validated Linear", "Polynomial"]
rmse_scores = [best_test_rmse, polynomial_rmse]
r2_scores = [best_test_r2, polynomial_r2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(model_names, rmse_scores)
axes[0].set_title("RMSE Comparison")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y")

axes[1].bar(model_names, r2_scores)
axes[1].set_title("R2 Comparison")
axes[1].set_ylabel("R2 Score")
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y")

plt.tight_layout()
plt.show()


## 12. Capture Model Metadata

Production teams need enough metadata to explain what was trained, when it was trained, and how it performed.


In [ ]:
model_metadata = {
    "model_name": "enterprise_credit_pricing_regression",
    "created_at": datetime.now().isoformat(),
    "training_rows": train_data.count(),
    "test_rows": test_data.count(),
    "features": feature_columns,
    "label": target_column,
    "best_model": {
        "rmse": float(best_test_rmse),
        "mae": float(best_test_mae),
        "r2": float(best_test_r2),
        "regParam": best_regression.getRegParam(),
        "elasticNetParam": best_regression.getElasticNetParam(),
    },
    "polynomial_model": {
        "degree": 2,
        "feature_count": polynomial_feature_count,
        "rmse": float(polynomial_rmse),
        "r2": float(polynomial_r2),
    },
}

print(json.dumps(model_metadata, indent=2))


## 13. SAS Reference: Saving Model Outputs

SAS workflows often save scored rows and model reports as datasets or ODS output.

```sas
PROC REG DATA=train_data OUTEST=model_parameters;
  MODEL interest_rate_percent = credit_score annual_income loan_amount debt_to_income
      credit_utilization delinquencies loan_term_months prior_default;
RUN;
QUIT;

PROC SCORE DATA=test_data SCORE=model_parameters OUT=scored_test TYPE=PARMS;
  VAR credit_score annual_income loan_amount debt_to_income credit_utilization
      delinquencies loan_term_months prior_default;
RUN;
```


## 14. Enterprise Migration Notes

| Need | SAS Pattern | Databricks Pattern |
| --- | --- | --- |
| Scale beyond one machine | Large SAS server | Spark cluster |
| Repeatable training | Stored programs/macros | ML pipelines |
| Parameter search | Manual PROC runs | `CrossValidator` |
| Nonlinear pricing | Manual squared columns | `PolynomialExpansion` |
| Model scoring | `PROC SCORE` | `model.transform()` |
| Model tracking | External documentation | MLflow or registry metadata |

Recommended migration path:
1. Start with a simple pricing model.
2. Match SAS output on a small sample.
3. Move feature engineering into a PySpark pipeline.
4. Add cross-validation after the baseline is trusted.
5. Add polynomial features when test metrics and risk logic support the extra complexity.
